In [1]:
# Install Gemini API library
!pip install -U google-generativeai

  Using cached google_ai_generativelanguage-0.6.15-py3-none-any.whl.metadata (5.7 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached uritemplate-4.2.0-py3-none-any.whl.metadata (2.6 kB)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
  Using cached grpcio_status-1.74.0-py3-none-any.whl.metadata (1.1 kB)
  Using cached grpcio_status-1.73.1-py3-none-any.whl.metadata (1.1 kB)
  Using cached grpcio_status-1.73.0-py3-none-any.whl.metadata (1.1 kB)
  Using cached grpcio_status-1.72.2-py3-none-any.whl.metadata (1.1 kB)
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
  Using cached grpcio_status-1.72.1-py3-none-any.whl.metadata (1.1 kB)
  Using cached grpcio_status-1.71.2-py3-none-any


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Imports and Inititalizations

In [11]:
import pandas as pd
import base64
import os
import google.generativeai as genai
from io import BytesIO
from PIL import Image
import nltk
import json
import itertools


env_path = ".env" 
if os.path.exists(env_path):
    with open(env_path, "r") as f:
        API_KEYS = [line.strip() for line in f if line.strip() and not line.startswith("#")]
else:
    print(f"Error: '{env_path}' file not found. Ensure it is in: {os.getcwd()}")
    API_KEYS = []

# 2. Create a generator that cycles through the keys infinitely
if API_KEYS:
    key_cycle = itertools.cycle(API_KEYS)
    print(f"Successfully loaded {len(API_KEYS)} API keys from .env for rotation.")
else:
    key_cycle = None
    print("Warning: No API keys loaded. The inference engine will fail.")

# Evaluator used in CC_OCR
from doc_parsing_evaluator import ParsingEvaluator

# Ensure NLTK data is available
nltk.download('punkt')

# Paths based on your local directory structure
BASE_DIR = r"D:\Projects\BTP\CC-OCR_Dataset\doc_parsing"
OUT_DIR = r"D:\Projects\BTP\Evaluation_Results"
os.makedirs(OUT_DIR, exist_ok=True)

# Updated model selection
MODEL_NAME = "gemma-3-27b-it"

# Initialize the evaluator
evaluator = ParsingEvaluator(group_name="OCR_Eval")

Successfully loaded 6 API keys from .env for rotation.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\khann\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## Inference Engine (Google cloud Models)

In [3]:
def query_gemini(image_bytes, task_type):
    """Sends image to Gemini API, rotating through available API keys"""
    
    # 3. Pull the next key and re-configure the API
    current_key = next(key_cycle)
    genai.configure(api_key=current_key)
    
    if task_type == "table":
        prompt = "Parse this table into a clean, structured HTML format with <table> tags. Include all row and column spans."
    else:
        prompt = "Parse this document into LaTeX format. Provide only the LaTeX code."

    # Convert bytes to a PIL Image
    img = Image.open(BytesIO(image_bytes))
    
    # Generate content using the current key
    try:
        model = genai.GenerativeModel(model_name=MODEL_NAME)
        response = model.generate_content([prompt, img])
        return response.text
    except Exception as e:
        print(f"Error with key {current_key[:10]}...: {e}")
        return ""

## Document Parsing Evaluation (LaTeX)

In [12]:
doc_tasks = {
    "Scanned": os.path.join(BASE_DIR, "doc", "doc_scan_eng_75.tsv"),
    "Photoed": os.path.join(BASE_DIR, "doc", "doc_photo_eng_75.tsv")
}

doc_results = {}

for label, path in doc_tasks.items():
    if not os.path.exists(path): continue
    df = pd.read_csv(path, sep='\t')
    scores = []
    
    print(f"Processing {label} documents...")
    
    for _, row in df.iterrows():
        img_bytes = base64.b64decode(row['image'])
        ground_truth = str(row['answer']).strip() 
        
        prediction = query_gemini(img_bytes, "doc")
        
        # FIX: Use the professional NED logic with LaTeX cleaning
        score = evaluator.evaluate_single_doc_sample(ground_truth, prediction)
        scores.append(score)
        
    doc_results[label] = sum(scores) / len(scores) if scores else 0

print(f"Document Parsing (NED with LaTeX) Results: {doc_results}")

Processing Scanned documents...
Error with key AIzaSyBT5X...: Invalid operation: The `response.text` quick accessor requires the response to contain a valid `Part`, but none were returned. The candidate's [finish_reason](https://ai.google.dev/api/generate-content#finishreason) is 4. Meaning that the model was reciting from copyrighted material.
Error with key AIzaSyBT5X...: Invalid operation: The `response.text` quick accessor requires the response to contain a valid `Part`, but none were returned. The candidate's [finish_reason](https://ai.google.dev/api/generate-content#finishreason) is 4. Meaning that the model was reciting from copyrighted material.
Error with key AIzaSyATn3...: Invalid operation: The `response.text` quick accessor requires the response to contain a valid `Part`, but none were returned. The candidate's [finish_reason](https://ai.google.dev/api/generate-content#finishreason) is 4. Meaning that the model was reciting from copyrighted material.
Processing Photoed docu

## Table Parsing Evaluation (HTML)

In [13]:
table_tasks = {
    "Scanned": os.path.join(BASE_DIR, "table", "table_scan_eng_75.tsv"),
    "Photoed": os.path.join(BASE_DIR, "table", "table_photo_eng_75.tsv")
}

table_results = {}

for label, path in table_tasks.items():
    if not os.path.exists(path): 
        print(f"Skipping {label}: Path not found.")
        continue
        
    df = pd.read_csv(path, sep='\t')
    scores = []
    
    print(f"Processing {label} tables...")
    
    for _, row in df.iterrows():
        img_bytes = base64.b64decode(row['image'])
        ground_truth_html = str(row['answer']).strip() 
        
        prediction = query_gemini(img_bytes, "table")
        
        # FIX: Use the official Tree Edit Distance (TEDS) metric
        score = evaluator.evaluate_single_table_sample(ground_truth_html, prediction)
        scores.append(score)
        
    table_results[label] = sum(scores) / len(scores) if scores else 0

print(f"\nTable Parsing (TEDS Results): {table_results}")

Processing Scanned tables...
Processing Photoed tables...

Table Parsing (TEDS Results): {'Scanned': 0.6555066174747228, 'Photoed': 0.5207718393466946}


## Final Report 

In [14]:
summary = {
    "Document Parsing (NED-LaTeX)": doc_results,
    "Table Parsing (HTML-Sim)": table_results,
    "Model": MODEL_NAME,
    "Track": "English Document Parsing"
}

with open(os.path.join(OUT_DIR, "stage1_final_report.json"), "w") as f:
    json.dump(summary, f, indent=4)

print("Evaluation Complete. Check 'stage1_final_report.json' for full details.")

Evaluation Complete. Check 'stage1_final_report.json' for full details.
